# 🤖 Competitive Intelligence with DeepAgents
## Archetype 1 of 2 — Autonomous Sub-Agent Delegation

---

This notebook researches 5 competitors in a product category using **LangChain DeepAgents**.

### How DeepAgents works here

```
Orchestrator Agent
├── write_todos()          ← Plans the research upfront
├── task(competitor_1)     ← Sub-agent 1: isolated context window
│     └── tavily_search()  ← Searches pricing, news, reviews
│     └── write_file()     ← Saves findings to disk
├── task(competitor_2)     ← Sub-agent 2: fresh context, no bleed
│     └── ...
├── task(competitor_N)     ← ...
└── read_file() × N        ← Reads all reports, synthesizes brief
```

**Key insight:** Each sub-agent lives in its own context window.  
The orchestrator never carries 5 competitors of raw research simultaneously.

---

### Metrics tracked (via LangGraph reporter)
- Total tokens (input + output)
- Estimated USD cost
- Number of LLM calls
- Number of Tavily searches
- Wall-clock execution time

> Compare these numbers to `02_prompt_chaining_competitive_intel.ipynb` to see the difference.

## 📦 1. Install Dependencies

In [1]:
%pip install -q deepagents langchain langchain-openai langchain-community \
    langgraph langchain-core tavily-python langchain-tavily \
    python-dotenv pandas rich matplotlib ipywidgets

Note: you may need to restart the kernel to use updated packages.


## 🔑 2. API Keys & Configuration

Set your keys below **or** create a `.env` file in this directory:
```
OPENAI_API_KEY=sk-...
TAVILY_API_KEY=tvly-...
```

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads .env if present

# Override here if not using .env
# os.environ["OPENAI_API_KEY"] = "sk-..."
# os.environ["TAVILY_API_KEY"] = "tvly-..."

assert os.environ.get("OPENAI_API_KEY"), "Missing OPENAI_API_KEY"
assert os.environ.get("TAVILY_API_KEY"), "Missing TAVILY_API_KEY"

# ── Tunable parameters ────────────────────────────────────────────────────
PRODUCT_CATEGORY = "AI coding assistants"   # ← Change this to any category
NUM_COMPETITORS  = 5                          # ← How many competitors to research
MODEL_NAME       = "gpt-5-mini"              # ← 500K TPM limit
MAX_SEARCH_RESULTS = 5                        # ← Tavily results per search
# ─────────────────────────────────────────────────────────────────────────

print(f"✅ Keys loaded | Category: {PRODUCT_CATEGORY} | Model: {MODEL_NAME}")

✅ Keys loaded | Category: AI coding assistants | Model: gpt-4o


## 📊 3. LangGraph Reporting Setup

We import the shared `TokenCostTracker` callback and `build_report` function from `utils/reporting.py`.  
Both notebooks use the **same reporter** so results are directly comparable.

In [3]:
import sys
sys.path.insert(0, ".")

from utils.reporting import TokenCostTracker, build_report, compare_runs

# Instantiate the tracker — attach this as a callback to every LLM call
tracker = TokenCostTracker(model_name=MODEL_NAME)
print("✅ LangGraph reporter ready")

✅ LangGraph reporter ready


## 🛠️ 4. Build the DeepAgent

`create_deep_agent` returns a **compiled LangGraph graph**.  
We give the agent:
- `TavilySearchResults` for web search
- Our `TokenCostTracker` as a callback so every LLM call is metered
- A system prompt scoped to competitive intelligence

In [4]:
from deepagents import create_deep_agent
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch

# ── Tools available to the agent and its sub-agents ──────────────────────
tavily_tool = TavilySearch(
    max_results=MAX_SEARCH_RESULTS,
    include_answer=True,
    include_raw_content=False,
)

# ── System prompt — scopes the agent to competitive intelligence ──────────
SYSTEM_PROMPT = f"""You are a competitive intelligence analyst specializing in \
the {PRODUCT_CATEGORY} market.

Your research methodology:
1. Plan your work upfront using write_todos.
2. For EACH competitor, spawn a dedicated sub-agent using `task` to keep \
research isolated and context windows lean.
3. Each sub-agent should:
   a. Search for the competitor's current pricing and positioning.
   b. Search for recent news (last 6 months).
   c. Search for user sentiment (Reddit, G2, HackerNews).
   d. Write a structured markdown report to a file: reports/<competitor>.md
4. After all sub-agents complete, read all report files and synthesize a \
single analyst brief with a recommendation matrix.

Format the final brief as markdown with these sections:
- Executive Summary
- Competitor Profiles (one subsection per competitor)
- Pricing Comparison Table
- Sentiment Summary
- Strategic Recommendations
- White Space Opportunities
"""

# ── Create the DeepAgent ──────────────────────────────────────────────────
model = init_chat_model(f"openai:{MODEL_NAME}")

agent = create_deep_agent(
    model=model,
    tools=[tavily_tool],
    system_prompt=SYSTEM_PROMPT,
)

print("✅ DeepAgent created — it is a compiled LangGraph graph")
print(f"   Graph type: {type(agent).__name__}")

✅ DeepAgent created — it is a compiled LangGraph graph
   Graph type: CompiledStateGraph


## 🚀 5. Run the Competitive Intelligence Research

We invoke the agent with a single high-level task and stream events.  
Watch the agent:
1. Write a plan (`write_todos`)
2. Spawn sub-agents for each competitor (`task`)
3. Each sub-agent search → write file
4. Orchestrator synthesize into final brief

> ⏱️ This will take several minutes depending on the number of competitors.

In [5]:
import time
from IPython.display import Markdown, display, clear_output

TASK = f"""
Research the top {NUM_COMPETITORS} competitors in the '{PRODUCT_CATEGORY}' space.

For each competitor find:
- Current pricing tiers and free-tier limits
- Key differentiating features
- Recent product announcements or pivots (last 6 months)
- User sentiment from Reddit, HackerNews, or G2
- Estimated market positioning (enterprise vs indie vs SMB)

Produce a comprehensive competitive intelligence brief with a strategic
recommendation matrix and identified white-space opportunities.
"""

print(f"🔬 Starting DeepAgent research...")
print(f"   Task: {TASK[:120].strip()}...\n")

tracker = TokenCostTracker(model_name=MODEL_NAME)  # fresh tracker
start_time = time.time()
final_result = None
step_count = 0

# Stream the agent's execution — each chunk is a LangGraph event
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": TASK}]},
    config={"callbacks": [tracker]},
    stream_mode="values",
):
    messages = chunk.get("messages", [])
    if messages:
        last = messages[-1]
        role = getattr(last, "type", "unknown")
        step_count += 1

        # Print a compact progress line for tool calls
        if hasattr(last, "tool_calls") and last.tool_calls:
            for tc in last.tool_calls:
                name = tc.get("name", "tool")
                args_preview = str(tc.get("args", ""))[:80]
                print(f"  🔧 [{step_count}] {name}: {args_preview}")
        elif role == "tool":
            content_preview = str(getattr(last, "content", ""))[:80]
            print(f"  📥 Tool result: {content_preview}")
        elif role == "ai" and getattr(last, "content", ""):
            final_result = last.content
            print(f"  🤖 Agent thinking...")

tracker.stop()
elapsed = time.time() - start_time
print(f"\n✅ Research complete in {elapsed:.1f}s")

🔬 Starting DeepAgent research...
   Task: Research the top 5 competitors in the 'AI coding assistants' space.

For each competitor find:
- Current pricing tiers...

  🔧 [3] write_todos: {'todos': [{'content': 'Identify the top 5 competitors in the AI coding assistan
  📥 Tool result: Updated todo list to [{'content': 'Identify the top 5 competitors in the AI codi
  🔧 [5] tavily_search: {'query': 'top AI coding assistant competitors 2023'}
  📥 Tool result: {"query": "top AI coding assistant competitors 2023", "follow_up_questions": nul
  🔧 [7] task: {'description': 'Conduct comprehensive research on GitHub Copilot. \n\nThe resea
  🔧 [7] task: {'description': 'Conduct comprehensive research on AWS CodeWhisperer. \n\nThe re
  🔧 [7] task: {'description': 'Conduct comprehensive research on OpenAI Codex. \n\nThe researc
  🔧 [7] task: {'description': 'Conduct comprehensive research on Tabnine. \n\nThe research sho
  🔧 [7] task: {'description': 'Conduct comprehensive research on Kite. \n\nThe r

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-kOz3ZE1ir4I1etm3Nu1pO3Xo on tokens per min (TPM): Limit 30000, Used 30000, Requested 765. Please try again in 1.53s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}

## 📄 6. Final Intelligence Brief

In [ ]:
if final_result:
    display(Markdown(final_result))
else:
    print("No final result captured — check agent output above.")

## 📊 7. LangGraph Cost & Token Report

The `build_report` function runs a small **LangGraph graph** that formats and prints the metrics.  
Save `tracker.summary()` to a variable — we'll use it in the comparison notebook.

In [ ]:
report = build_report(
    tracker=tracker,
    approach="DeepAgents",
    result_text=final_result or "",
)

# Save summary for cross-notebook comparison
deepagents_summary = tracker.summary()
print("\n📋 Summary dict (use in comparison notebook):")
import json
print(json.dumps(deepagents_summary, indent=2))

## 📈 8. Visualize Token Distribution

A breakdown of prompt vs. completion tokens across all LLM calls.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Token usage pie chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f"DeepAgents — Token Usage\n{PRODUCT_CATEGORY}", fontsize=14, fontweight="bold")

# Pie: prompt vs completion tokens
ax1 = axes[0]
labels = ["Prompt Tokens", "Completion Tokens"]
sizes = [tracker.prompt_tokens, tracker.completion_tokens]
colors = ["#4A90E2", "#E25A4A"]
if sum(sizes) > 0:
    ax1.pie(sizes, labels=labels, colors=colors, autopct="%1.1f%%", startangle=90)
else:
    ax1.text(0.5, 0.5, "No token data\n(run cell 5 first)", ha="center", va="center")
ax1.set_title("Token Split")

# Bar: step-by-step token usage across LLM calls
ax2 = axes[1]
llm_steps = [s for s in tracker.steps if s["type"] == "llm"]
if llm_steps:
    x = range(len(llm_steps))
    pt = [s["prompt_tokens"] for s in llm_steps]
    ct = [s["completion_tokens"] for s in llm_steps]
    ax2.bar(x, pt, label="Prompt", color="#4A90E2", alpha=0.85)
    ax2.bar(x, ct, bottom=pt, label="Completion", color="#E25A4A", alpha=0.85)
    ax2.set_xlabel("LLM Call #")
    ax2.set_ylabel("Tokens")
    ax2.set_title("Tokens Per LLM Call")
    ax2.legend()
else:
    ax2.text(0.5, 0.5, "No step data", ha="center", va="center")
    ax2.set_title("Tokens Per LLM Call")

plt.tight_layout()
plt.savefig("deepagents_tokens.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Chart saved to deepagents_tokens.png")

## 🗂️ 9. Inspect Sub-Agent Files

DeepAgents writes competitor research to `reports/` directory.  
Let's list and preview what was produced.

In [ ]:
import os
from pathlib import Path

reports_dir = Path("reports")
if reports_dir.exists():
    files = list(reports_dir.glob("*.md"))
    print(f"📁 Found {len(files)} competitor report(s) in reports/:\n")
    for f in sorted(files):
        size = f.stat().st_size
        print(f"  • {f.name:40s}  ({size:,} bytes)")

    if files:
        print(f"\n--- Preview: {files[0].name} ---")
        display(Markdown(files[0].read_text()[:2000] + "\n\n*[truncated...]*"))
else:
    print("ℹ️  No reports/ directory found — agent may have stored output differently.")

## ✅ 10. Summary

| What DeepAgents did | Why it matters |
|---|---|
| Created a plan with `write_todos` | Task visibility, can debug/resume steps |
| Spawned `N` isolated sub-agents | Each competitor's research = fresh context, no contamination |
| Sub-agents wrote to files | Parent agent never carries raw research in its window |
| Orchestrator synthesized from files | Synthesis context = just the summaries, not all raw data |
| LangGraph streams every step | Full observability on token use and tool calls |

---

**Next:** Open `02_prompt_chaining_competitive_intel.ipynb` to run the same task  
using sequential prompt chaining and compare the metrics.